[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pyshka501/rl-textbook/blob/main/translations/ru/notebooks/ch14_practical_rlhf_ru.ipynb)

<div style="background: linear-gradient(135deg, #944454 0%, #1B474D 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">Глава 14. Практический RLHF</h1>
  <p style="color: #BCE2E7; margin-top: 10px; font-size: 1.1em;">
    Сквозной RLHF на TRL: обучение модели вознаграждения, запуск PPO, мониторинг KL-дивергенции
    и вознаграждения, требования к вычислениям при масштабировании.
  </p>
</div>

**Цели обучения:**
- С помощью TRL запустить минимальный RLHF-пайплайн с GPT-2
- Обучить голову модели вознаграждения на данных предпочтений
- Запустить циклы PPO-обучения и логировать KL-дивергенцию, вознаграждение и длину ответа
- Интерпретировать диагностические графики: компромисс вознаграждение vs KL
- Понять, как масштабируются вычисления при разных размерах модели

<div style="background: #1C1B19; border-left: 4px solid #20808D; padding: 15px; border-radius: 6px; margin: 10px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">Подготовка окружения</h2>
  <p style="color: #CDCCCA; margin-top: 8px;">Настоятельно рекомендуется T4 GPU. Цикл PPO займёт ~5 минут на GPU и ~25 минут на CPU. Включите GPU: Runtime → Change runtime type → T4 GPU.</p>
</div>

In [ ]:
!pip install -q transformers accelerate datasets trl torch matplotlib numpy tqdm

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer, AutoTokenizer, AutoModelForCausalLM
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead, create_reference_model
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from tqdm.auto import tqdm
import warnings, time, math, copy
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

plt.rcParams.update({
    'figure.facecolor': '#F7F6F2', 'axes.facecolor': '#F9F8F5',
    'axes.edgecolor': '#D4D1CA', 'axes.labelcolor': '#28251D',
    'text.color': '#28251D', 'xtick.color': '#7A7974',
    'ytick.color': '#7A7974', 'grid.color': '#D4D1CA',
    'font.family': 'sans-serif',
})

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§14.1 Обзор пайплайна RLHF</h2>
</div>

Полный RLHF-пайплайн состоит из трёх этапов:

```
Этап 1: SFT             Этап 2: Модель вознаграждения   Этап 3: RL-дообучение
──────────────────      ──────────────────────────────   ────────────────────────────────
Базовая LM              SFT-модель + линейная голова     Цикл PPO / GRPO / DPO:
  +                     обучается на парных предпочтениях   стратегия генерирует y
Инструкции          →   потеря Брэдли\,---\,Терри       →   модель вознаграждения даёт r(x,y)
  SFT-модель            модель вознагр. r_φ(x,y)             KL-штраф против референса
                                                              обновление по policy gradient
```

В этой главе реализуем **этапы 2 и 3** через TRL, с GPT-2 в качестве базовой модели.

In [ ]:
# ── Stage 2: Train a Reward Model ────────────────────────────────────────
# We train a GPT-2 model with a linear reward head on synthetic preference data.

PREF_DATA = [
    ('What is reinforcement learning?',
     'Reinforcement learning is a framework where an agent learns optimal behaviour by receiving rewards from an environment through trial and error.',
     'I do not know.'),
    ('Explain transformers.',
     'Transformers use self-attention to process sequences in parallel, enabling long-range dependency modelling with high efficiency.',
     'They are neural networks.'),
    ('What is gradient descent?',
     'Gradient descent minimises a loss function by iteratively updating parameters in the direction of the steepest decrease, scaled by a learning rate.',
     'It updates things.'),
    ('Describe RLHF.',
     'RLHF (Reinforcement Learning from Human Feedback) fine-tunes language models using human preferences as a reward signal, typically via PPO.',
     'It is a training method.'),
    ('What is overfitting?',
     'Overfitting occurs when a model learns spurious patterns in training data that do not generalise, leading to high test error despite low training error.',
     'Bad results on test.'),
    ('Define attention mechanism.',
     'Attention computes a weighted sum of value vectors, where the weights are derived from the compatibility of a query with each key.',
     'It pays attention to parts.'),
    ('What is a policy in RL?',
     'A policy is a mapping from states to actions (deterministic) or distributions over actions (stochastic), defining the agent behaviour.',
     'Something the agent does.'),
    ('Explain KL divergence.',
     'KL divergence D_KL(P||Q) measures how much distribution P diverges from Q, computed as the expected log-ratio of their probability densities.',
     'A divergence measure.'),
]


class RewardModelGPT2(nn.Module):
    """GPT-2 with a scalar reward head (takes mean of last hidden states)."""
    def __init__(self, model_name='gpt2'):
        super().__init__()
        self.backbone = GPT2LMHeadModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.n_embd
        self.reward_head = nn.Linear(hidden_size, 1)
        nn.init.zeros_(self.reward_head.weight)
        nn.init.zeros_(self.reward_head.bias)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone.transformer(input_ids=input_ids, attention_mask=attention_mask)
        # Use hidden state at last non-padding position
        hidden = outputs.last_hidden_state  # (B, T, H)
        # Gather the last real token position per batch item
        lengths = attention_mask.sum(dim=1) - 1  # (B,)
        last_hidden = hidden[torch.arange(hidden.size(0)), lengths]  # (B, H)
        reward = self.reward_head(last_hidden).squeeze(-1)  # (B,)
        return reward


tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

def tokenize_text(prompt, response, max_len=80):
    text = f"{prompt} {response}{tokenizer.eos_token}"
    enc = tokenizer(text, truncation=True, max_length=max_len,
                    padding='max_length', return_tensors='pt')
    return enc.input_ids.squeeze(), enc.attention_mask.squeeze()


class PrefDataset(Dataset):
    def __init__(self, data, repeat=6):
        self.data = data * repeat
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        p, c, r = self.data[idx]
        c_ids, c_mask = tokenize_text(p, c)
        r_ids, r_mask = tokenize_text(p, r)
        return c_ids, c_mask, r_ids, r_mask

rm_dataset = PrefDataset(PREF_DATA)
rm_loader  = DataLoader(rm_dataset, batch_size=4, shuffle=True)

reward_model = RewardModelGPT2().to(DEVICE)
rm_optimizer = AdamW(reward_model.parameters(), lr=1e-5)

rm_losses = []
N_EPOCHS_RM = 4

for epoch in range(N_EPOCHS_RM):
    reward_model.train()
    ep_loss = 0.0
    for c_ids, c_mask, r_ids, r_mask in rm_loader:
        c_ids, c_mask = c_ids.to(DEVICE), c_mask.to(DEVICE)
        r_ids, r_mask = r_ids.to(DEVICE), r_mask.to(DEVICE)

        r_chosen   = reward_model(c_ids, c_mask)
        r_rejected = reward_model(r_ids, r_mask)

        # Bradley-Terry loss
        loss = -F.logsigmoid(r_chosen - r_rejected).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
        rm_optimizer.step()
        rm_optimizer.zero_grad()

        rm_losses.append(loss.item())
        ep_loss += loss.item()

    print(f'RM Epoch {epoch+1}: avg loss = {ep_loss/len(rm_loader):.4f}')

print('Reward model training complete.')

In [ ]:
# Verify reward model: score chosen > rejected
reward_model.eval()
print('Reward model sanity check:')
correct = 0
with torch.no_grad():
    for prompt, chosen, rejected in PREF_DATA:
        c_ids, c_mask = tokenize_text(prompt, chosen)
        r_ids, r_mask = tokenize_text(prompt, rejected)
        rc = reward_model(c_ids.unsqueeze(0).to(DEVICE), c_mask.unsqueeze(0).to(DEVICE)).item()
        rr = reward_model(r_ids.unsqueeze(0).to(DEVICE), r_mask.unsqueeze(0).to(DEVICE)).item()
        ok = rc > rr
        correct += int(ok)
        print(f'  chosen={rc:+.3f}  rejected={rr:+.3f}  correct={ok}')

print(f'\nAccuracy: {correct}/{len(PREF_DATA)} = {correct/len(PREF_DATA)*100:.0f}%')

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§14.2 PPO с TRL</h2>
</div>

`PPOTrainer` из TRL оркестрирует RLHF-цикл:
1. **Генерируем** ответы стратегии.
2. **Оцениваем** их моделью вознаграждения.
3. **Считаем KL-штраф** $D_{\text{KL}}[\pi_\theta \| \pi_{\text{ref}}]$ относительно замороженной референсной модели.
4. **Обновляем** стратегию clipped-целью PPO.

Полное вознаграждение для оптимизации стратегии:
$$ r_{\text{total}}(x, y) = r_\phi(x, y) - \beta \, \text{KL}(\pi_\theta \| \pi_{\text{ref}}) $$

In [ ]:
# ── PPO Training with TRL ─────────────────────────────────────────────────
# We simulate the TRL PPO loop manually for pedagogical transparency,
# logging the same metrics TRL would log.

PPO_MODEL_NAME = 'gpt2'
BETA_KL = 0.05          # KL penalty coefficient
N_PPO_STEPS = 60        # number of policy update steps
BATCH_SIZE_PPO = 4
MAX_NEW_TOKENS = 30
LR_PPO = 1e-5
CLIP_EPS = 0.2

# Policy model (will be updated)
policy_model = GPT2LMHeadModel.from_pretrained(PPO_MODEL_NAME).to(DEVICE)
# Reference model (frozen copy)
ref_model = GPT2LMHeadModel.from_pretrained(PPO_MODEL_NAME).to(DEVICE)
for p in ref_model.parameters():
    p.requires_grad = False
ref_model.eval()

ppo_optimizer = AdamW(policy_model.parameters(), lr=LR_PPO)

# Prompts for PPO generation
PPO_PROMPTS = [
    "Explain what machine learning is:",
    "What is a neural network?",
    "Describe the transformer model:",
    "What is reinforcement learning?",
    "Explain overfitting:",
    "What is gradient descent?",
    "How does attention work?",
    "Describe RLHF:",
]


def compute_sequence_log_prob(model, input_ids, response_ids):
    """Compute log-prob of response_ids conditioned on input_ids."""
    full_ids = torch.cat([input_ids, response_ids], dim=1)
    with torch.no_grad():
        out = model(full_ids)
    logits = out.logits[:, input_ids.shape[1]-1:-1, :]  # shift to predict response
    log_probs = F.log_softmax(logits, dim=-1)
    token_lp = log_probs.gather(2, response_ids.unsqueeze(-1)).squeeze(-1)
    return token_lp.sum(dim=-1)  # (B,)


# Logging containers
log_rewards     = []
log_kl          = []
log_ppo_loss    = []
log_resp_length = []
log_total_reward = []

t0 = time.time()
print(f'Starting PPO loop ({N_PPO_STEPS} steps)...')

for step in range(N_PPO_STEPS):
    # Sample a batch of prompts
    batch_prompts = np.random.choice(PPO_PROMPTS, BATCH_SIZE_PPO, replace=True)
    policy_model.eval()

    batch_rewards = []
    batch_kl      = []
    batch_lengths = []
    batch_response_ids_list = []
    batch_input_ids_list    = []
    batch_old_log_probs     = []

    for prompt in batch_prompts:
        enc = tokenizer(prompt, return_tensors='pt', padding=True).to(DEVICE)
        prompt_ids = enc.input_ids

        # Generate response
        with torch.no_grad():
            gen_out = policy_model.generate(
                prompt_ids,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.9,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        resp_ids = gen_out[:, prompt_ids.shape[1]:]  # response only
        resp_text = tokenizer.decode(resp_ids[0], skip_special_tokens=True)

        # Reward from reward model
        full_text = prompt + ' ' + resp_text + tokenizer.eos_token
        rm_enc = tokenizer(full_text, truncation=True, max_length=80,
                           padding='max_length', return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            r = reward_model(rm_enc.input_ids, rm_enc.attention_mask).item()

        # KL divergence estimate (per token)
        with torch.no_grad():
            full_ids = torch.cat([prompt_ids, resp_ids], dim=1)
            pol_out = policy_model(full_ids)
            ref_out = ref_model(full_ids)
            pol_lp = F.log_softmax(pol_out.logits[:, :-1, :], dim=-1)
            ref_lp = F.log_softmax(ref_out.logits[:, :-1, :], dim=-1)
            kl_per_token = (pol_lp.exp() * (pol_lp - ref_lp)).sum(-1)  # (B, T-1)
            kl_mean = kl_per_token[0, prompt_ids.shape[1]-1:].mean().item()

        # Old log-prob (for PPO ratio)
        with torch.no_grad():
            old_lp = compute_sequence_log_prob(policy_model, prompt_ids, resp_ids).item()

        batch_rewards.append(r)
        batch_kl.append(kl_mean)
        batch_lengths.append(resp_ids.shape[1])
        batch_response_ids_list.append(resp_ids)
        batch_input_ids_list.append(prompt_ids)
        batch_old_log_probs.append(old_lp)

    # Compute total reward (penalised by KL)
    total_rewards = [r - BETA_KL * kl for r, kl in zip(batch_rewards, batch_kl)]

    # PPO policy gradient update
    policy_model.train()
    ppo_step_loss = 0.0
    for i in range(BATCH_SIZE_PPO):
        prompt_ids = batch_input_ids_list[i]
        resp_ids   = batch_response_ids_list[i]
        old_lp     = batch_old_log_probs[i]
        adv        = total_rewards[i]  # scalar advantage

        new_lp = compute_sequence_log_prob(policy_model, prompt_ids, resp_ids)
        # PPO clipped ratio
        ratio = torch.exp(new_lp - old_lp)
        surr1 = ratio * adv
        surr2 = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv
        loss = -torch.min(surr1, surr2).mean()
        ppo_step_loss += loss.item()

        (loss / BATCH_SIZE_PPO).backward()

    torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 0.5)
    ppo_optimizer.step()
    ppo_optimizer.zero_grad()

    log_rewards.append(np.mean(batch_rewards))
    log_kl.append(np.mean(batch_kl))
    log_ppo_loss.append(ppo_step_loss / BATCH_SIZE_PPO)
    log_resp_length.append(np.mean(batch_lengths))
    log_total_reward.append(np.mean(total_rewards))

    if (step + 1) % 10 == 0:
        elapsed = time.time() - t0
        print(f'Step {step+1:3d}/{N_PPO_STEPS}: '
              f'reward={log_rewards[-1]:.3f}, '
              f'KL={log_kl[-1]:.4f}, '
              f'len={log_resp_length[-1]:.1f}, '
              f'elapsed={elapsed:.0f}s')

print(f'\nPPO training complete in {time.time()-t0:.1f}s')

In [ ]:
# ── Diagnostic Plots ─────────────────────────────────────────────────────
smooth = lambda x, w=5: np.convolve(x, np.ones(w)/w, mode='valid')
steps = np.arange(N_PPO_STEPS)

fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])
ax4 = fig.add_subplot(gs[1, 0])
ax5 = fig.add_subplot(gs[1, 1:3])

# 1. Reward over time
ax1.plot(steps, log_rewards, color='#BCE2E7', alpha=0.5, linewidth=1)
ax1.plot(range(4, N_PPO_STEPS), smooth(log_rewards), color='#20808D', linewidth=2)
ax1.set_xlabel('PPO Step')
ax1.set_ylabel('Reward Model Score')
ax1.set_title('Reward Over Training', fontweight='bold')
ax1.grid(alpha=0.4)

# 2. KL divergence over time
ax2.plot(steps, log_kl, color='#FFC553', alpha=0.5, linewidth=1)
ax2.plot(range(4, N_PPO_STEPS), smooth(log_kl), color='#A84B2F', linewidth=2)
ax2.set_xlabel('PPO Step')
ax2.set_ylabel('KL(π_θ || π_ref)')
ax2.set_title('KL Divergence Over Training', fontweight='bold')
ax2.grid(alpha=0.4)

# 3. Response length
ax3.plot(steps, log_resp_length, color='#7A39BB', alpha=0.5, linewidth=1)
ax3.plot(range(4, N_PPO_STEPS), smooth(log_resp_length),
         color='#7A39BB', linewidth=2)
ax3.set_xlabel('PPO Step')
ax3.set_ylabel('Avg Response Length (tokens)')
ax3.set_title('Response Length Over Training', fontweight='bold')
ax3.grid(alpha=0.4)

# 4. PPO loss
ax4.plot(steps, log_ppo_loss, color='#006494', alpha=0.5, linewidth=1)
ax4.plot(range(4, N_PPO_STEPS), smooth(log_ppo_loss),
         color='#006494', linewidth=2)
ax4.set_xlabel('PPO Step')
ax4.set_ylabel('PPO Clipped Loss')
ax4.set_title('PPO Loss', fontweight='bold')
ax4.grid(alpha=0.4)

# 5. Reward vs KL scatter (key diagnostic)
scatter = ax5.scatter(log_kl, log_rewards, c=steps, cmap='viridis', alpha=0.7, s=50)
cbar = plt.colorbar(scatter, ax=ax5)
cbar.set_label('Training Step')
ax5.set_xlabel('KL Divergence from Reference')
ax5.set_ylabel('Reward Model Score')
ax5.set_title('§14.2 — Reward vs KL Divergence (Colour = Training Step)', fontweight='bold')
ax5.grid(alpha=0.4)

# Add annotation
ax5.annotate('Early training\n(low KL)', xy=(np.percentile(log_kl, 25), np.mean(log_rewards[:15])),
             xytext=(np.percentile(log_kl, 40), np.mean(log_rewards[:15]) - 0.1),
             arrowprops=dict(arrowstyle='->', color='#7A7974'), fontsize=9, color='#7A7974')

fig.suptitle('Chapter 14 — Practical RLHF: Training Diagnostics (GPT-2 + PPO)',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('ch14_fig1_rlhf_diagnostics.png', dpi=120, bbox_inches='tight')
plt.show()

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§14.3 Качество генерации до и после PPO</h2>
</div>

In [ ]:
# Compare base GPT-2 vs PPO-tuned GPT-2
base_model = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE)
base_model.eval()
policy_model.eval()

test_prompts = [
    "Explain what machine learning is:",
    "Describe the transformer model:",
]

def generate_text(model, prompt, max_new=50):
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new,
                             do_sample=True, temperature=0.7, top_p=0.9,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)

def get_reward_score(prompt, response):
    text = prompt + ' ' + response + tokenizer.eos_token
    enc = tokenizer(text, truncation=True, max_length=80,
                    padding='max_length', return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        return reward_model(enc.input_ids, enc.attention_mask).item()

print('=' * 70)
for prompt in test_prompts:
    base_resp = generate_text(base_model, prompt)
    ppo_resp  = generate_text(policy_model, prompt)
    base_score = get_reward_score(prompt, base_resp)
    ppo_score  = get_reward_score(prompt, ppo_resp)

    print(f'\nPrompt: {prompt}')
    print(f'  [Base GPT-2] (reward={base_score:+.3f}):\n    {base_resp[:150]}')
    print(f'  [PPO GPT-2]  (reward={ppo_score:+.3f}):\n    {ppo_resp[:150]}')
    improvement = ppo_score - base_score
    print(f'  Reward improvement: {improvement:+.3f}')
    print('-' * 70)

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§14.4 Требования к вычислениям при разных размерах моделей</h2>
</div>

Стоимость обучения растёт примерно как $6ND$ FLOPs за прямой+обратный проход, где $N$ — число параметров, $D$ — число обучающих токенов. RLHF требует одновременного запуска **нескольких моделей** (стратегия + референс + модель вознаграждения), увеличивая требования к памяти примерно втрое.

In [ ]:
# ── Compute scaling analysis ───────────────────────────────────────────────
import math

model_sizes = {
    'GPT-2 small (124M)':   1.24e8,
    'GPT-2 medium (345M)':  3.45e8,
    'GPT-2 XL (1.5B)':      1.5e9,
    'TinyLlama (1.1B)':     1.1e9,
    'Llama 2 (7B)':         7e9,
    'Llama 3.1 (8B)':       8e9,
    'Llama 3.1 (70B)':      7e10,
}

# Memory estimates for inference (policy + ref + reward) in GB
# Approx: bytes_per_param * 3 models (fp16 = 2 bytes)
bytes_per_param_fp16 = 2
n_models_rlhf = 3  # policy + ref + reward head

# Typical RLHF tokens for one epoch
rlhf_tokens = 1e9  # 1B tokens
flops_per_token_per_param = 6  # forward + backward

# A100 GPU: ~312 TFLOPS for fp16
a100_tflops = 312e12
# T4 GPU: ~65 TFLOPS
t4_tflops = 65e12

rows = []
for name, params in model_sizes.items():
    mem_gb = params * bytes_per_param_fp16 * n_models_rlhf / 1e9
    total_flops = flops_per_token_per_param * params * rlhf_tokens
    t4_hours = total_flops / t4_tflops / 3600
    a100_hours = total_flops / a100_tflops / 3600
    fits_t4   = '✓' if mem_gb < 14 else '✗'  # T4 has ~16 GB
    fits_a100 = '✓' if mem_gb < 78 else '✗'  # A100 80 GB
    rows.append((name, f'{params/1e9:.2f}B', f'{mem_gb:.1f}',
                 f'{t4_hours:.0f}h', f'{a100_hours:.1f}h', fits_t4, fits_a100))

print(f'{"Model":<30} {"Params":>8} {"RLHF Mem (GB)":>14} {"T4 Est.":>10} {"A100 Est.":>10} {"T4 Fits":>8} {"A100":>6}')
print('-' * 90)
for r in rows:
    print(f'{r[0]:<30} {r[1]:>8} {r[2]:>14} {r[3]:>10} {r[4]:>10} {r[5]:>8} {r[6]:>6}')

In [ ]:
# Visualise compute scaling
names = list(model_sizes.keys())
params_list = [v for v in model_sizes.values()]
mem_gb_list = [p * bytes_per_param_fp16 * n_models_rlhf / 1e9 for p in params_list]
a100_hrs_list = [flops_per_token_per_param * p * rlhf_tokens / a100_tflops / 3600
                 for p in params_list]

short_names = [n.split('(')[0].strip() for n in names]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Memory
bar_colors = ['#20808D' if m < 14 else '#FFC553' if m < 78 else '#A84B2F'
              for m in mem_gb_list]
axes[0].barh(range(len(names)), mem_gb_list, color=bar_colors)
axes[0].axvline(14, color='#20808D', linestyle='--', linewidth=2, label='T4 limit (~14 GB)')
axes[0].axvline(78, color='#FFC553', linestyle='--', linewidth=2, label='A100 limit (~78 GB)')
axes[0].set_yticks(range(len(names)))
axes[0].set_yticklabels(short_names, fontsize=9)
axes[0].set_xlabel('RLHF Memory (GB) — policy + ref + reward model (fp16)')
axes[0].set_title('§14.4 — Memory Requirements for RLHF', fontweight='bold')
axes[0].legend(fontsize=9)

# Compute (A100 hours)
axes[1].barh(range(len(names)), [math.log10(max(h, 0.001)) for h in a100_hrs_list],
             color='#1B474D')
axes[1].set_yticks(range(len(names)))
axes[1].set_yticklabels(short_names, fontsize=9)
axes[1].set_xlabel('Log₁₀(A100 GPU-Hours) for 1B token RLHF run')
axes[1].set_title('Estimated Compute Time (A100)', fontweight='bold')

for i, (h, p) in enumerate(zip(a100_hrs_list, params_list)):
    label = f'{h:.1f}h' if h < 1000 else f'{h/24:.0f}d'
    axes[1].text(math.log10(max(h, 0.001)) + 0.02, i, label, va='center', fontsize=8)

plt.tight_layout()
plt.savefig('ch14_fig2_compute_scaling.png', dpi=120, bbox_inches='tight')
plt.show()

<div style="background: #1C1B19; border-left: 4px solid #20808D; padding: 20px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0 0 12px 0;">Сводка и ключевые выводы</h2>
  <ul style="color: #D4D1CA; line-height: 1.8; margin: 0 0 16px 0; padding-left: 22px;">
    <li><strong>Модели вознаграждения</strong> обучаются с потерей Брэдли\,---\,Терри на парах предпочтений и превращают человеческие сравнения в скалярную обратную связь $r_\phi(x, y)$.</li>
    <li><strong>Цикл PPO</strong> циклически генерирует ответы, оценивает их моделью вознаграждения, применяет KL-штраф к референсной стратегии и обновляет стратегию.</li>
    <li><strong>KL-дивергенция</strong> обычно растёт во время обучения; $\beta$ управляет тем, насколько сильно может расти вознаграждение, прежде чем стратегия слишком уйдёт от референса.</li>
    <li><strong>Вознаграждение vs KL</strong> — главный баланс RLHF: лучшая стратегия не та, у которой максимум прокси-награды, а та, что повышает вознаграждение, оставаясь достаточно близко к референсному распределению.</li>
    <li><strong>Требования к вычислениям</strong> быстро растут, потому что RLHF держит в памяти стратегию, референс и модель вознаграждения; GPT-2-масштаб помещается на T4, а модели от 7B обычно требуют multi-GPU установок класса A100.</li>
  </ul>

  <h3 style="color: #BCE2E7; margin: 0 0 8px 0;">Полный рецепт RLHF на практике</h3>
  <ol style="color: #D4D1CA; line-height: 1.8; margin: 0 0 16px 0; padding-left: 22px;">
    <li><strong>SFT</strong> дообучает базовую модель на инструкционных данных.</li>
    <li><strong>Моделирование вознаграждения</strong> обучает модель Брэдли\,---\,Терри на парах предпочтений людей.</li>
    <li><strong>RL-дообучение</strong> запускает PPO или GRPO с KL-регуляризацией.</li>
    <li><strong>Проверки выравнивания</strong> отслеживают вознаграждение, KL, разнообразие ответов и переоптимизацию вознаграждения.</li>
    <li><strong>DPO</strong> может заменить моделирование вознаграждения и RL-дообучение, когда нужен более простой путь обучения по предпочтениям.</li>
  </ol>

  <p style="color: #D4D1CA; line-height: 1.8; margin: 0; font-style: italic;">
    Это завершает RLHF-раздел учебника. По продакшен-реализациям — см. <a href="https://huggingface.co/docs/trl" style="color: #BCE2E7;">документацию TRL</a> и DeepSpeed ZeRO для multi-GPU.
  </p>
</div>